# tsauditor Example: Auditing Video Pose Keypoint Time Series

A non-finance, non-sensor use case: auditing per-person pose keypoint sequences extracted from video (YOLOv8-pose + tracking) for an action recognition pipeline. Each tracked person becomes one panel entity; their per-frame skeleton (17 COCO keypoints, 34 x/y values) is the time series.

This notebook uses **synthetic data** so it runs standalone. The real case study that motivated this example is described in the markdown cells below.


## 1. Generate synthetic pose data

Simulates the same shape as real extracted data: some tracks with natural jittery motion, and, deliberately, a few tracks with an injected **boundary-clamp artifact** (a keypoint pinned to a constant value for many consecutive frames), matching the real bug this example is based on.


In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)
NUM_KEYPOINTS = 17
FPS = 15

def make_natural_track(n_frames, track_id):
    """Normal case: every joint jitters naturally frame to frame."""
    base = np.random.uniform(-1, 1, size=(NUM_KEYPOINTS, 2))
    kp = base[None, :, :] + np.random.normal(0, 0.02, size=(n_frames, NUM_KEYPOINTS, 2)).cumsum(axis=0) * 0.1
    return kp

def make_clamped_track(n_frames, track_id, clamp_joint=15, clamp_value=1.0, clamp_start=10, clamp_len=40):
    """Injects the real bug: one joint's y-coordinate gets pinned to a
    constant value for a stretch of frames, mimicking YOLOv8-pose
    clamping an off-screen keypoint to the frame boundary."""
    kp = make_natural_track(n_frames, track_id)
    end = min(clamp_start + clamp_len, n_frames)
    kp[clamp_start:end, clamp_joint, 1] = clamp_value  # y-coordinate pinned
    return kp

frames = {}

# 15 well-behaved tracks
for i in range(15):
    n = np.random.randint(40, 90)
    kp = make_natural_track(n, i)
    rows = [{"frame_idx": t, **{f"kp{j}_{ax}": kp[t, j, k]
             for j in range(NUM_KEYPOINTS) for k, ax in enumerate("xy")}}
            for t in range(n)]
    df = pd.DataFrame(rows)
    df["timestamp"] = pd.Timestamp("2000-01-01") + pd.to_timedelta(df["frame_idx"] / FPS, unit="s")
    frames[f"natural_track_{i}"] = df.set_index("timestamp").drop(columns=["frame_idx"])

# 5 tracks with the injected boundary-clamp artifact
for i in range(5):
    n = np.random.randint(50, 90)
    kp = make_clamped_track(n, i)
    rows = [{"frame_idx": t, **{f"kp{j}_{ax}": kp[t, j, k]
             for j in range(NUM_KEYPOINTS) for k, ax in enumerate("xy")}}
            for t in range(n)]
    df = pd.DataFrame(rows)
    df["timestamp"] = pd.Timestamp("2000-01-01") + pd.to_timedelta(df["frame_idx"] / FPS, unit="s")
    frames[f"clamped_track_{i}"] = df.set_index("timestamp").drop(columns=["frame_idx"])

print(f"{len(frames)} tracks generated ({15} natural, {5} with injected clamp artifact).")


20 tracks generated (15 natural, 5 with injected clamp artifact).


## 2. Audit each track in parallel

With ~20 short entities (this toy example) or thousands (a real extraction run), `group_col` on one combined long-format frame hits real per-task overhead once entities are numerous and short. Using separate per-entity DataFrames with the documented joblib pattern scales better for this shape of data.

`target=None` is deliberate: there's no per-frame prediction target here (a track's label, if any, is one scalar for the whole sequence, not per-row), so leakage checks are correctly skipped, not worked around.


In [4]:
from joblib import Parallel, delayed
import tsauditor as tsa

def audit_track(track_id, df):
    report = tsa.scan(df, target=None, domain="sensor",
                       run_leakage=False, run_stationarity=False)
    return track_id, report

results = dict(Parallel(n_jobs=-1)(
    delayed(audit_track)(tid, df) for tid, df in frames.items()
))


In [2]:
!pip install tsauditor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.8/82.8 kB 3.6 MB/s eta 0:00:00


## 3. Collect ANO001 (stuck value) findings

In [5]:
stuck = []
for track_id, report in results.items():
    for issue in (report.critical + report.warnings + report.info):
        if "ANO001" in str(getattr(issue, "code", "")):
            stuck.append((track_id, issue))

print(f"{len(stuck)} ANO001 findings across {len(frames)} tracks:\n")
for track_id, issue in stuck:
    print(f"  {track_id}: {issue}")


5 ANO001 findings across 20 tracks:

  clamped_track_0: Issue(module='anomaly', code='ANO001', severity='warning', description='Stuck values detected.', column='kp15_y', evidence={'max_stuck_duration': 40}, group=None)
  clamped_track_1: Issue(module='anomaly', code='ANO001', severity='warning', description='Stuck values detected.', column='kp15_y', evidence={'max_stuck_duration': 40}, group=None)
  clamped_track_2: Issue(module='anomaly', code='ANO001', severity='warning', description='Stuck values detected.', column='kp15_y', evidence={'max_stuck_duration': 40}, group=None)
  clamped_track_3: Issue(module='anomaly', code='ANO001', severity='warning', description='Stuck values detected.', column='kp15_y', evidence={'max_stuck_duration': 40}, group=None)
  clamped_track_4: Issue(module='anomaly', code='ANO001', severity='warning', description='Stuck values detected.', column='kp15_y', evidence={'max_stuck_duration': 40}, group=None)


Notice it correctly catches the 5 `clamped_track_*` entities (and correctly leaves the 15 `natural_track_*` entities alone), exactly matching what happened on the real dataset this example is based on.


## 4. The real case study

This synthetic example is modeled on a real bug found and fixed while building an unrelated project: a fight and agitation detection pipeline for CCTV monitoring, intended to eventually support detecting sudden aggression in dementia and Parkinson's patients in care settings.

### Background

That pipeline extracts human pose keypoints from video using YOLOv8-pose, tracks each person across frames with ByteTrack, and classifies each tracked person's motion sequence with an LSTM as aggressive or calm. Roughly 3600 tracked people were extracted from about 2000 video clips (the RLVS dataset). Each tracked person's keypoints were normalized (centered on hip midpoint, scaled by shoulder width) and cached as a sequence for training.

### Why tsuditor was applied here

tsauditor was originally built for financial and sensor time series, not video pose data. It was applied to this pipeline's output anyway, because the underlying data shape is the same regardless of domain: each tracked person is an independent panel entity, and their per-frame keypoints form a time series for that entity. The question TSAuditor answers, is anything in this time series behaving in a way that shouldn't happen, applies just as well to a skeleton's joint position over time as it does to a stock's price over time.

The specific check of interest was ANO001, stuck value detection. The concern was a known failure mode in this kind of pipeline: if pose tracking silently stalls (loses the person but keeps reporting a value) rather than failing cleanly, the resulting sequence looks like real data but isn't. A frame where every keypoint is exactly zero is easy to catch and already was, but a subtler stall, where the model keeps reporting a plausible-looking but frozen position, is much harder to catch by eye and was not something the pipeline's existing checks looked for.

### What the audit found

Each of the roughly 3600 tracked people's keypoint sequences was converted into its own DataFrame and audited independently, in parallel, using the pattern shown in section 2 of this notebook. The scan flagged 1099 ANO001 findings. Nearly all of them were on vertical (y-axis) coordinate columns, and several tracks showed a joint holding the exact same value for over 50 consecutive frames, out of tracks that were themselves often only 50 to 90 frames long. In other words, some tracked people had a joint that never moved for essentially the entire time they were tracked.

### Investigating the finding

The first hypothesis was tracking failure: perhaps ByteTrack was losing the person and re-detecting a static background object under the same ID. This was ruled out by inspecting the raw, pre-normalization keypoint values for a flagged track directly, rather than only looking at the normalized, already processed data. The raw dump showed something more specific:

```
frame 0: kp7(elbow)=[168.95  224]  kp11(hip)=[122.72  224]  kp15(ankle)=[117.84  224]
frame 1: kp7(elbow)=[166.73  224]  kp11(hip)=[122.44  224]  kp15(ankle)=[114.75  224]
frame 2: kp7(elbow)=[165.50  224]  kp11(hip)=[120.85  224]  kp12(hip)=[114.13  224]
...
```

The x-coordinates were changing naturally, frame to frame, consistent with a real, moving person. The y-coordinates, across several unrelated joints (elbow, both hips, ankle), were pinned to exactly 224. Checking the source video's actual pixel height confirmed it: 224 pixels. That is not a coincidence, it is the frame boundary.

### Root cause

YOLOv8-pose, when a tracked person's body extends beyond the visible edge of the frame (a very common situation in close-up or partially cropped CCTV and street-level footage), does not mark the corresponding keypoint as undetected. Instead, it reports a keypoint clamped to the frame boundary, as if that were a normal, confident detection. Because the reported value is a real, non-zero coordinate rather than (0, 0), it passes straight through a simple all-zero detection-failure filter undetected. It also looks fine in a visual sanity check, since a skeleton plot of a single frame with a boundary-clamped ankle still looks like a plausible human pose. The problem only becomes visible when checking whether a value that should vary over time is actually varying, which is exactly what TSAuditor's stuck value check does.

### Fix

The pipeline's normalization step was updated to reject any frame where a keypoint sits within about one pixel of the frame boundary, in addition to the existing all-zero check. This is a targeted fix aimed specifically at the confirmed cause, rather than a broader change to tracking or detection logic.

### Verification

The full dataset was re-extracted with the fix in place, and the identical tsauditor scan was re-run against the corrected data. The result: zero ANO001 findings, across 3345 tracked people (down from 3617, since the tracks that consisted mostly of the fake boundary-clamped signal are now correctly excluded rather than silently retained). This before-and-after comparison, same check, same underlying pipeline, only the fix applied in between, is what confirms the fix actually resolved the underlying issue rather than only hiding its symptom.

### Takeaway

ANO001 is not a finance-specific or sensor-specific check. The underlying idea, a value that is supposed to vary should not be constant, applies equally to a stock price, a temperature reading, or a joint's position in a video. The specific failure mode found here (a computer vision model clamping instead of failing) looks nothing like a typical financial data quality issue, but it produces the exact same statistical signature that tsauditor was already built to catch.
